In [1]:
# ============================================
# CLASSIFICATION: SI > median
# Google Colab notebook
# ============================================

# Если работаешь в Colab, сначала загрузи файл xlsx вручную:
# from google.colab import files
# uploaded = files.upload()

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)


# 2. Загрузка данных

sheet_id = "1q-nbWuFrfrIBMXmZfNW78N3bx5v60Vb9"
url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv"

df = pd.read_csv(url)


# 2.1 Базовая предобработка


# Удаляю служебный индексный столбец
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

print("Размер после удаления служебного столбца:", df.shape)

# Проверяю пропуски
print("\nКоличество пропусков:")
print(df.isna().sum().sum())

# Если вдруг пропуски есть — заполняю медианами
df = df.fillna(df.median(numeric_only=True))


# 3. Создание целевой переменной


# Медиана SI по всей выборке
si_median = df["SI"].median()
print(f"\nМедиана SI: {si_median:.6f}")

# Бинарный таргет:
# 1 -> SI > median
# 0 -> SI <= median
df["SI_median_class"] = (df["SI"] > si_median).astype(int)

print("\nРаспределение классов:")
print(df["SI_median_class"].value_counts())
print(df["SI_median_class"].value_counts(normalize=True))


# 4. Формирование признаков



# Для задачи предсказания SI нельзя оставлять в признаках сам SI,
# а также IC50 и CC50, потому что SI вычисляется через них.
drop_cols = ["SI", "IC50, mM", "CC50, mM", "SI_median_class"]

X = df.drop(columns=drop_cols, errors="ignore")
y = df["SI_median_class"]

print("\nРазмер X:", X.shape)
print("Размер y:", y.shape)


# 5. Train / Test split


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTrain shape:", X_train.shape)
print("Test shape:", X_test.shape)


# 6. Модель 1 — Logistic Regression


logreg_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=5000, random_state=42))
])

logreg_model.fit(X_train, y_train)
y_pred_logreg = logreg_model.predict(X_test)

print("=" * 60)
print("Logistic Regression")
print("=" * 60)
print("Accuracy:", round(accuracy_score(y_test, y_pred_logreg), 4))
print("F1-score:", round(f1_score(y_test, y_pred_logreg), 4))
print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred_logreg))
print("\nClassification report:")
print(classification_report(y_test, y_pred_logreg))


# 7. Модель 2 — Random Forest


rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)

rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

print("=" * 60)
print("Random Forest")
print("=" * 60)
print("Accuracy:", round(accuracy_score(y_test, y_pred_rf), 4))
print("F1-score:", round(f1_score(y_test, y_pred_rf), 4))
print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred_rf))
print("\nClassification report:")
print(classification_report(y_test, y_pred_rf))


# 8. Сводная таблица результатов


results = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_logreg),
        accuracy_score(y_test, y_pred_rf)
    ],
    "F1-score": [
        f1_score(y_test, y_pred_logreg),
        f1_score(y_test, y_pred_rf)
    ]
})

print("=" * 60)
print("Сравнение моделей")
print("=" * 60)
display(results.sort_values(by="F1-score", ascending=False).reset_index(drop=True))

best_model_name = results.sort_values(by="F1-score", ascending=False).iloc[0]["Model"]
print(f"Лучшая модель по F1-score: {best_model_name}")


# 9. Важность признаков для Random Forest


feature_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": rf_model.feature_importances_
}).sort_values(by="importance", ascending=False)

print("\nТоп-15 важных признаков:")
display(feature_importance.head(15))


Размер после удаления служебного столбца: (1001, 213)

Количество пропусков:
36

Медиана SI: 3.846154

Распределение классов:
SI_median_class
0    501
1    500
Name: count, dtype: int64
SI_median_class
0    0.5005
1    0.4995
Name: proportion, dtype: float64

Размер X: (1001, 210)
Размер y: (1001,)

Train shape: (800, 210)
Test shape: (201, 210)
Logistic Regression
Accuracy: 0.6219
F1-score: 0.6275

Confusion matrix:
[[61 40]
 [36 64]]

Classification report:
              precision    recall  f1-score   support

           0       0.63      0.60      0.62       101
           1       0.62      0.64      0.63       100

    accuracy                           0.62       201
   macro avg       0.62      0.62      0.62       201
weighted avg       0.62      0.62      0.62       201

Random Forest
Accuracy: 0.6517
F1-score: 0.6196

Confusion matrix:
[[74 27]
 [43 57]]

Classification report:
              precision    recall  f1-score   support

           0       0.63      0.73      0.68 

,Model,Accuracy,F1-score
0,Logistic Regression,0.621891,0.627451
1,Random Forest,0.651741,0.619565


Лучшая модель по F1-score: Logistic Regression

Топ-15 важных признаков:


,feature,importance
99,VSA_EState4,0.018743
21,BCUT2D_CHGLO,0.017234
1,MaxEStateIndex,0.016510
25,BCUT2D_MRLOW,0.015596
0,MaxAbsEStateIndex,0.015439
3,MinEStateIndex,0.015121
19,BCUT2D_MWLOW,0.014936
23,BCUT2D_LOGPLOW,0.014705
22,BCUT2D_LOGPHI,0.014672
105,FractionCSP3,0.014476


## Выводы

В этой части я решала задачу бинарной классификации: определяла, превышает ли значение **SI** медиану по выборке.

Сначала я загрузила датасет и выполнила базовую предобработку. После удаления служебного столбца размер таблицы составил **1001 строку и 213 признаков**. Также я проверила данные на пропуски: всего было обнаружено **36 пропущенных значений**, после чего они были заполнены медианными значениями.

Далее я рассчитала медиану целевого признака **SI**, она составила **3.846154**. На основе этого значения я сформировала бинарную целевую переменную **SI_median_class**:
- класс **1** — если **SI > 3.846154**
- класс **0** — если **SI ≤ 3.846154**

Распределение классов получилось практически сбалансированным:
- класс **0** — **501 объектов** (**50.05%**)
- класс **1** — **500 объектов** (**49.95%**)

После исключения из признаков столбцов **SI**, **IC50**, **CC50** и целевой переменной размер матрицы признаков составил **(1001, 210)**. Затем я разделила данные на обучающую и тестовую выборки:
- **train:** **800 объектов**, **210 признаков**
- **test:** **201 объект**, **210 признаков**

Далее я обучила две модели: **Logistic Regression** и **Random Forest**, а затем сравнила их по метрикам качества.

### Logistic Regression

Модель **Logistic Regression** показала следующие результаты:
- **Accuracy:** **0.6219**
- **F1-score:** **0.6275**

Матрица ошибок:
```text
[[61 40]
 [36 64]]

Из classification report:

для класса 0:
precision = 0.63
recall = 0.60
f1-score = 0.62
для класса 1:
precision = 0.62
recall = 0.64
f1-score = 0.63

Итоговая accuracy по тестовой выборке 0.62.

Random Forest

Модель Random Forest показала следующие результаты:

Accuracy: 0.6517
F1-score: 0.6196

Матрица ошибок:

[[74 27]
 [43 57]]

Из classification report:

для класса 0:
precision = 0.63
recall = 0.73
f1-score = 0.68
для класса 1:
precision = 0.68
recall = 0.57
f1-score = 0.62

Итоговая accuracy по тестовой выборке 0.65.

Сравнение моделей

По итогам сравнения моделей я получила следующие результаты:

Logistic Regression
Accuracy: 0.621891
F1-score: 0.627451
Random Forest
Accuracy: 0.651741
F1-score: 0.619565

Несмотря на то, что Random Forest показал более высокую Accuracy, лучшей моделью по F1-score стала Logistic Regression. Поскольку в задаче классификации мне важно учитывать баланс между точностью и полнотой, в качестве основной модели я выбрала  Logistic Regression.



Дополнительно я проанализировала важность признаков, рассчитанную моделью Random Forest. Наиболее значимыми оказались следующие дескрипторы:

VSA_EState4 — 0.018743
BCUT2D_CHGLO — 0.017234
MaxEStateIndex — 0.016510
BCUT2D_MRLOW — 0.015596
MaxAbsEStateIndex — 0.015439
MinEStateIndex — 0.015121
BCUT2D_MWLOW — 0.014936
BCUT2D_LOGPLOW — 0.014705
BCUT2D_LOGPHI — 0.014672
FractionCSP3 — 0.014476
qed — 0.012946
MinAbsEStateIndex — 0.012755
BertzCT — 0.012741
FpDensityMorgan3 — 0.012568
BCUT2D_CHGHI — 0.012516
Итоговый вывод

В результате я получила рабочее решение для задачи SI > median. Обе модели показали умеренное качество, что говорит о том, что задача не является тривиальной и требует дальнейшего улучшения.

Я считаю, что на текущем этапе Logistic Regression является более предпочтительной моделью, потому что она показала лучший F1-score = 0.6275, а значит, лучше сбалансировала качество предсказания двух классов.